In [ ]:
import warnings
warnings.filterwarnings(
    action="ignore")
warnings.filterwarnings("error", category=FutureWarning,
                        module=r"torch\.nn\.modules\.module")


In [ ]:
# exp_sgd_svm.py  ── plain-SGD, default SVM loss
import subprocess, sys, time, statistics, pandas as pd
from pathlib import Path

# ──────────────────────────────────────────────────────────────────────────────
BEST_CFG = {
    "Cornell"     : {"epochs":10, "lr_scale":0.1},
    "Dermatology" : {"epochs":10, "lr_scale":0.5},
    "HHAR"        : {"epochs":30, "lr_scale":0.1},
    "ISOLET"      : {"epochs":30, "lr_scale":2.0},
    "ORL"         : {"epochs":30, "lr_scale":0.1},
    "USPS"        : {"epochs":20, "lr_scale":0.5},
    "Vehicle"     : {"epochs":10, "lr_scale":1.0},
}

OPTIMIZER   = "sgd"
LOSS_ARG    = []                
EPSILONS    = [1.0, 2.0, 4.0, 8.0]
N_REPEATS   = 5
OUT_DIR     = Path("exp_results_sgd_svm"); OUT_DIR.mkdir(exist_ok=True)

# ──────────────────────────────────────────────────────────────────────────────
def run_once(ds, eps, epc, lr, seed=None):
    cmd = [
        "python", "main-opacus.py",
        "--data", ds,
        "--optimizer", OPTIMIZER,
        "--num_epoch", str(epc),
        "--lr_scale", str(lr),
        "--eps", str(eps),
    ] + LOSS_ARG                      
    if seed is not None:
        cmd += ["--seed", str(seed)] 

    print(" ".join(cmd))
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)
    out, err = p.communicate()
    if err:  print(err, file=sys.stderr)

    acc = None
    for ln in out.splitlines():
        low = ln.lower()
        if "test accuracy" in low:
            tokens = [s for s in low.replace('%','').split() if s.replace('.','',1).isdigit()]
            if tokens:
                acc = float(tokens[-1])
                break
    if acc is None:
        raise RuntimeError(f"accuracy not found for {ds}, eps={eps}")
    return acc
# ──────────────────────────────────────────────────────────────────────────────
def main():
    rows = []
    for ds, cfg in BEST_CFG.items():
        for eps in EPSILONS:
            acc_list = []
            for rep in range(N_REPEATS):
                acc = run_once(ds, eps, cfg["epochs"], cfg["lr_scale"], seed=None)
                acc_list.append(acc)
                time.sleep(0.5)
            mean = statistics.mean(acc_list)
            std  = statistics.stdev(acc_list)
            rows.append({
                "dataset": ds,
                "epsilon": eps,
                "mean": mean,
                "std": std,
                "latex": f"{mean:.3f}$\\pm${std:.3f}"
            })
            pd.DataFrame(rows).to_csv(OUT_DIR/"summary_progress.csv", index=False)
            print(f"[{ds:11s} | ε={eps:>4}] {rows[-1]['latex']}")

    df = pd.DataFrame(rows)
    df.to_csv(OUT_DIR/"summary_final.csv", index=False)
    print("\n▶ Summary:", OUT_DIR/"summary_final.csv")

    # LaTeX 행 출력
    print("\n============= LaTeX Rows =============")
    for ds in BEST_CFG.keys():
        vals = df.query("dataset == @ds").sort_values("epsilon")["latex"].tolist()
        print(f"{ds} & " + " & ".join(vals) + r" \\")
# ──────────────────────────────────────────────────────────────────────────────
if __name__ == "__main__":
    main()


In [ ]:
#!/usr/bin/env python
# -*- coding: utf-8 -*-
"""
exp_sgd_svm.py ── DP-SGD + SVM loss + LR-decay
"""

import subprocess
import sys
import time
import re
import statistics
import pandas as pd
from pathlib import Path

# ──────────────────────────────────────────────────────────────────────────────
BEST_CFG = {
    "Cornell":     {"epochs": 10, "lr_scale": 0.1},
    "Dermatology": {"epochs": 10, "lr_scale": 0.5},
    "HHAR":        {"epochs": 30, "lr_scale": 0.1},
    "ISOLET":      {"epochs": 30, "lr_scale": 2.0},
    "ORL":         {"epochs": 30, "lr_scale": 0.1},
    "USPS":        {"epochs": 20, "lr_scale": 0.5},
    "Vehicle":     {"epochs": 10, "lr_scale": 1.0},
}

OPTIMIZER  = "sgd"
LOSS_ARG   = []                           #
EPSILONS   = [1.0, 2.0, 4.0, 8.0]
N_REPEATS  = 5
OUT_DIR    = Path("exp_results_sgd_svm")
OUT_DIR.mkdir(exist_ok=True)

# ──────────────────────────────────────────────────────────────────────────────
def run_once(
    ds: str,
    eps: float,
    epc: int,
    lr: float,
    seed: int | None = None,
    use_lr_decay: bool = True,
) -> float:
    """main-opacus-decay.py """
    cmd = [
        "python", "main-opacus-decay.py",
        "--data",       ds,
        "--optimizer",  OPTIMIZER,
        "--num_epoch",  str(epc),
        "--lr_scale",   str(lr),
        "--eps",        str(eps),
    ] + LOSS_ARG

    if use_lr_decay:
        cmd.append("--lr_decay")

    if seed is not None:
        cmd += ["--seed", str(seed)]

    print(" ".join(cmd))
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE,
                            stderr=subprocess.PIPE, text=True)
    out, err = proc.communicate()
    if err:
        print(err, file=sys.stderr)

    acc = None
    for ln in out.splitlines():
        m = re.search(r"\bacc=([0-9.]+)", ln)
        if m:
            acc = float(m.group(1))  
    if acc is None:
        raise RuntimeError(f"accuracy not found for {ds}, eps={eps}")
    return acc

# ──────────────────────────────────────────────────────────────────────────────
def main() -> None:
    rows = []
    for ds, cfg in BEST_CFG.items():
        for eps in EPSILONS:
            acc_list = []
            for rep in range(N_REPEATS):
                acc = run_once(ds, eps, cfg["epochs"], cfg["lr_scale"], seed=None)
                acc_list.append(acc)

            mean = statistics.mean(acc_list)
            std  = statistics.stdev(acc_list)
            rows.append({
                "dataset": ds,
                "epsilon": eps,
                "mean":    mean,
                "std":     std,
                "latex":   f"{mean:.3f}$\\pm${std:.3f}"
            })
            pd.DataFrame(rows).to_csv(OUT_DIR / "summary_progress.csv", index=False)
            print(f"[{ds:11s} | ε={eps:>4}] {rows[-1]['latex']}")

    df = pd.DataFrame(rows)
    df.to_csv(OUT_DIR / "summary_final.csv", index=False)
    print("\n▶ Summary:", OUT_DIR / "summary_final.csv")

    print("\n============= LaTeX Rows =============")
    for ds in BEST_CFG.keys():
        vals = df.query("dataset == @ds").sort_values("epsilon")["latex"].tolist()
        print(f"{ds} & " + " & ".join(vals) + r" \\")
# ──────────────────────────────────────────────────────────────────────────────
if __name__ == "__main__":
    main()
